In [0]:
%run ./99_Config

In [0]:
%run ./98_Utilities

In [0]:
%run ./97_Logger

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

pipeline_start("04_Data_Validation")

# COMMAND ----------
bronze_path = f"{BRONZE_PATH}/transactions"
validated_path = f"{VALIDATED_PATH}/transactions"
rejected_path = f"{REJECTED_PATH}/transactions"

bronze_df = read_delta(bronze_path)

rows_read = record_count(bronze_df)

# COMMAND ----------
window_spec = Window.partitionBy("transaction_id").orderBy(col("load_timestamp").desc())

df = bronze_df.withColumn("row_num", row_number().over(window_spec))

df = (
    df.withColumn(
        "rejection_reason",
        when(col("transaction_id").isNull(),"Transaction ID Missing")
        .when(col("customer_id").isNull(),"Customer ID Missing")
        .when(col("account_number").isNull(),"Account Number Missing")
        .when(col("amount").isNull(),"Amount Missing")
        .when(col("amount") < MIN_TRANSACTION_AMOUNT,"Invalid Amount")
        .when(col("amount") > MAX_TRANSACTION_AMOUNT,"Amount Exceeds Limit")
        .when(col("currency").isNull(),"Currency Missing")
        .when(col("transaction_timestamp").isNull(),"Timestamp Missing")
        .when(~col("transaction_type").isin(VALID_TRANSACTION_TYPES),"Invalid Transaction Type")
        .when(col("row_num") > 1,"Duplicate Transaction")
    )
)


In [0]:
valid_df = (
    df.filter(col("rejection_reason").isNull())
      .drop("row_num","rejection_reason")
)

invalid_df = (
    df.filter(col("rejection_reason").isNotNull())
      .drop("row_num")
)

valid_rows = record_count(valid_df)
rejected_rows = record_count(invalid_df)

In [0]:
write_delta(valid_df, validated_path, "overwrite")
write_delta(invalid_df, rejected_path, "overwrite")

create_delta_table("validated_transactions", validated_path)
create_delta_table("rejected_transactions", rejected_path)

In [0]:
log_success(
    notebook="04_Data_Validation",
    dataset="transactions",
    rows_read=rows_read,
    rows_written=valid_rows,
    execution_time="Completed"
)

display_summary(
    "transactions",
    rows_read,
    valid_rows,
    rejected_rows
)

In [0]:
print("="*80)
print("DATA VALIDATION SUMMARY")
print("="*80)
print(f"Rows Read      : {rows_read}")
print(f"Valid Rows     : {valid_rows}")
print(f"Rejected Rows  : {rejected_rows}")
print("="*80)

pipeline_end("04_Data_Validation")